# EMS741 - Reptile Few-Shot Segmentation (with Checkpointing)

## 1. Environment and data download

In [1]:
# Check for GPU (works in Colab; harmless elsewhere)
!nvidia-smi -L || echo 'No GPU visible'

GPU 0: Tesla T4 (UUID: GPU-7fd4c28c-11fa-35eb-094b-fbe22b1cce0b)


In [2]:
# Download and extract EMS741 coursework data from Zenodo
import os, zipfile, urllib.request

DATA_URL = 'https://zenodo.org/records/18745413/files/ems741_cw_data.zip?download=1'
ZIP_PATH = 'data.zip'

if not os.path.exists(ZIP_PATH):
    print('Downloading data...')
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    print('Download complete.')

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall('./')

print('Train tasks:', os.listdir('./train'))
print('Val tasks:', os.listdir('./val'))
print('Test tasks:', os.listdir('./test'))

Download complete.
Train tasks: ['task_3', 'task_2', 'task_5', 'task_7']
Val tasks: ['task_6', 'task_4']
Test tasks: ['task_1', 'task_8']


## 2. Imports and basic configuration

In [3]:
import glob, random, json, csv
from datetime import datetime
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


## 3. Dataset utilities with mask handling

In [4]:
class SliceDataset(Dataset):
    def __init__(self, task_root, min_mask_sum=1.0, allow_empty=True):
        img_dir = os.path.join(task_root, 'images')
        msk_dir = os.path.join(task_root, 'masks')
        img_files = sorted(glob.glob(os.path.join(img_dir, '*.png')))
        msk_files = sorted(glob.glob(os.path.join(msk_dir, '*.png')))
        assert len(img_files) == len(msk_files), 'Mismatch between images and masks'
        pairs = []
        for ip, mp in zip(img_files, msk_files):
            m = np.array(Image.open(mp), dtype=np.float32)
            if m.max() > 1:
                m = m / 255.0
            if (m.sum() >= min_mask_sum) or allow_empty:
                pairs.append((ip, mp))
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        ip, mp = self.pairs[idx]
        img = np.array(Image.open(ip), dtype=np.float32)
        msk = np.array(Image.open(mp), dtype=np.float32)
        if img.max() > 1:
            img = img / 255.0
        if msk.max() > 1:
            msk = msk / 255.0
        img = torch.from_numpy(img).float().unsqueeze(0)
        msk = torch.from_numpy(msk).float().unsqueeze(0)
        return img, msk

## 4. Simple U-Net segmentation model (PyTorch)

In [5]:
class SimpleUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.c1a = nn.Conv2d(in_channels, 32, 3, padding=1)
        self.c1b = nn.Conv2d(32, 32, 3, padding=1)
        self.p1 = nn.MaxPool2d(2)
        self.c2a = nn.Conv2d(32, 64, 3, padding=1)
        self.c2b = nn.Conv2d(64, 64, 3, padding=1)
        self.p2 = nn.MaxPool2d(2)
        self.c3a = nn.Conv2d(64, 128, 3, padding=1)
        self.c3b = nn.Conv2d(128, 128, 3, padding=1)
        self.p3 = nn.MaxPool2d(2)
        self.c4a = nn.Conv2d(128, 256, 3, padding=1)
        self.c4b = nn.Conv2d(256, 256, 3, padding=1)
        self.up1 = nn.Upsample(scale_factor=2, mode='nearest')
        self.u1a = nn.Conv2d(256 + 128, 128, 3, padding=1)
        self.u1b = nn.Conv2d(128, 128, 3, padding=1)
        self.up2 = nn.Upsample(scale_factor=2, mode='nearest')
        self.u2a = nn.Conv2d(128 + 64, 64, 3, padding=1)
        self.u2b = nn.Conv2d(64, 64, 3, padding=1)
        self.up3 = nn.Upsample(scale_factor=2, mode='nearest')
        self.u3a = nn.Conv2d(64 + 32, 32, 3, padding=1)
        self.u3b = nn.Conv2d(32, 32, 3, padding=1)
        self.out = nn.Conv2d(32, out_channels, 1)
        self.act = nn.ReLU(inplace=True)
        self.final = nn.Sigmoid() if out_channels == 1 else nn.Softmax(dim=1)
    def forward(self, x):
        x1 = self.act(self.c1a(x)); x1 = self.act(self.c1b(x1)); p1 = self.p1(x1)
        x2 = self.act(self.c2a(p1)); x2 = self.act(self.c2b(x2)); p2 = self.p2(x2)
        x3 = self.act(self.c3a(p2)); x3 = self.act(self.c3b(x3)); p3 = self.p3(x3)
        b = self.act(self.c4a(p3)); b = self.act(self.c4b(b))
        y = self.up1(b); y = torch.cat([y, x3], dim=1); y = self.act(self.u1a(y)); y = self.act(self.u1b(y))
        y = self.up2(y); y = torch.cat([y, x2], dim=1); y = self.act(self.u2a(y)); y = self.act(self.u2b(y))
        y = self.up3(y); y = torch.cat([y, x1], dim=1); y = self.act(self.u3a(y)); y = self.act(self.u3b(y))
        return self.final(self.out(y))

## 5. Task listing and episode sampling

In [6]:
TRAIN_ROOT = './train'
VAL_ROOT   = './val'
TEST_ROOT  = './test'

def list_tasks(root):
    return sorted([os.path.join(root, d) for d in os.listdir(root) if d.startswith('task_')])

train_tasks = list_tasks(TRAIN_ROOT)
val_tasks   = list_tasks(VAL_ROOT)
test_tasks  = list_tasks(TEST_ROOT)

print('Train tasks:', train_tasks)
print('Val tasks:', val_tasks)
print('Test tasks:', test_tasks)

Train tasks: ['./train/task_2', './train/task_3', './train/task_5', './train/task_7']
Val tasks: ['./val/task_4', './val/task_6']
Test tasks: ['./test/task_1', './test/task_8']


In [7]:
def sample_task_episode(task_root, k_support=4, k_query=4, allow_empty=True, min_mask_sum=1.0):
    ds = SliceDataset(task_root, min_mask_sum=min_mask_sum, allow_empty=allow_empty)
    n = len(ds)
    if n == 0:
        raise ValueError(f'No usable slices in {task_root}')
    idxs = np.random.permutation(n)
    k_total = min(n, k_support + k_query)
    k_s = min(k_support, k_total // 2)
    k_q = min(k_query, k_total - k_s)
    idx_support = idxs[:k_s]
    idx_query   = idxs[k_s:k_s + k_q]
    xs, ys, xq, yq = [], [], [], []
    for i in idx_support:
        x, y = ds[i]; xs.append(x); ys.append(y)
    for i in idx_query:
        x, y = ds[i]; xq.append(x); yq.append(y)
    xs = torch.stack(xs)
    ys = torch.stack(ys)
    xq = torch.stack(xq) if len(xq) > 0 else None
    yq = torch.stack(yq) if len(yq) > 0 else None
    return xs, ys, xq, yq

## 6. Reptile inner-loop update

In [8]:
import copy

def inner_update(model, x_s, y_s, steps=5, lr_inner=1e-3, device=device):
    fast_model = copy.deepcopy(model).to(device)
    opt = torch.optim.SGD(fast_model.parameters(), lr=lr_inner)
    x_s = x_s.to(device)
    y_s = y_s.to(device)
    for _ in range(steps):
        opt.zero_grad()
        out = fast_model(x_s)
        if out.shape != y_s.shape:
            y_s = y_s.view_as(out)
        loss = F.binary_cross_entropy(out, y_s)
        loss.backward()
        opt.step()
    return fast_model

## 7. Reptile outer-loop with checkpointing

In [9]:
checkpoint_dir = 'checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

meta_config = {
    'meta_iters': 200,
    'tasks_per_meta_batch': 4,
    'k_support': 4,
    'k_query': 4,
    'lr_inner': 1e-3,
    'lr_meta': 0.1,
    'allow_empty_meta': False,
    'min_mask_sum': 1.0,
}

def save_checkpoint(model, meta_loss_history, it):
    ckpt_path = os.path.join(checkpoint_dir, f'meta_iter_{it}.pt')
    torch.save({
        'model_state': model.state_dict(),
        'meta_loss_history': meta_loss_history,
        'config': meta_config,
        'iter': it,
    }, ckpt_path)
    print('Saved checkpoint:', ckpt_path)


def reptile_meta_train(model, train_tasks, device=device):
    model.to(device)
    model.train()
    meta_loss_history = []
    start_iter = 0
    # OPTIONAL: load latest checkpoint if exists
    existing = [f for f in os.listdir(checkpoint_dir) if f.startswith('meta_iter_')]
    if existing:
        latest = sorted(existing, key=lambda x: int(x.split('_')[-1].split('.')[0]))[-1]
        ckpt = torch.load(os.path.join(checkpoint_dir, latest), map_location=device)
        model.load_state_dict(ckpt['model_state'])
        meta_loss_history = ckpt['meta_loss_history']
        start_iter = ckpt['iter'] + 1
        print('Resuming from checkpoint', latest, 'starting at iter', start_iter)
    for it in range(start_iter, meta_config['meta_iters']):
        meta_losses = []
        for p in model.parameters():
            p.grad = torch.zeros_like(p.data)
        used_tasks = random.sample(train_tasks, k=min(meta_config['tasks_per_meta_batch'], len(train_tasks)))
        for task_root in used_tasks:
            xs, ys, xq, yq = sample_task_episode(task_root, k_support=meta_config['k_support'], k_query=meta_config['k_query'], allow_empty=meta_config['allow_empty_meta'], min_mask_sum=meta_config['min_mask_sum'])
            fast_model = inner_update(model, xs, ys, steps=5, lr_inner=meta_config['lr_inner'], device=device)
            if xq is not None and yq is not None:
                xq = xq.to(device); yq = yq.to(device)
                with torch.no_grad():
                    out_q = fast_model(xq)
                    if out_q.shape != yq.shape:
                        yq = yq.view_as(out_q)
                    loss_q = F.binary_cross_entropy(out_q, yq)
                    meta_losses.append(loss_q.item())
            for p, p_fast in zip(model.parameters(), fast_model.parameters()):
                p.grad.data.add_(p_fast.data - p.data)
        for p in model.parameters():
            p.grad.data.div_(len(used_tasks))
            p.data.add_(meta_config['lr_meta'] * p.grad.data)
        mean_loss = float('nan') if len(meta_losses) == 0 else np.mean(meta_losses)
        meta_loss_history.append(mean_loss)
        if (it + 1) % 10 == 0:
            print(f'[Meta {it+1}/{meta_config['meta_iters']}] mean query BCE: {mean_loss:.4f}')
        if (it + 1) % 50 == 0:
            save_checkpoint(model, meta_loss_history, it+1)
    return meta_loss_history

meta_model = SimpleUNet(in_channels=1, out_channels=1)
meta_loss_history = reptile_meta_train(meta_model, train_tasks, device=device)

[Meta 10/200] mean query BCE: 0.6854
[Meta 20/200] mean query BCE: 0.6840
[Meta 30/200] mean query BCE: 0.6827
[Meta 40/200] mean query BCE: 0.6815
[Meta 50/200] mean query BCE: 0.6799
Saved checkpoint: checkpoints/meta_iter_50.pt
[Meta 60/200] mean query BCE: 0.6787
[Meta 70/200] mean query BCE: 0.6772
[Meta 80/200] mean query BCE: 0.6759
[Meta 90/200] mean query BCE: 0.6748
[Meta 100/200] mean query BCE: 0.6738
Saved checkpoint: checkpoints/meta_iter_100.pt
[Meta 110/200] mean query BCE: 0.6726
[Meta 120/200] mean query BCE: 0.6711
[Meta 130/200] mean query BCE: 0.6699
[Meta 140/200] mean query BCE: 0.6683
[Meta 150/200] mean query BCE: 0.6674
Saved checkpoint: checkpoints/meta_iter_150.pt
[Meta 160/200] mean query BCE: 0.6660
[Meta 170/200] mean query BCE: 0.6652
[Meta 180/200] mean query BCE: 0.6632
[Meta 190/200] mean query BCE: 0.6620
[Meta 200/200] mean query BCE: 0.6615
Saved checkpoint: checkpoints/meta_iter_200.pt


## 8. Few-shot adaptation and evaluation (store results only once)

In [10]:
def dice_score(pred, target, eps=1e-6):
    p = (pred > 0.5).float()
    t = target.float()
    inter = (p * t).sum(dim=(1,2,3))
    union = p.sum(dim=(1,2,3)) + t.sum(dim=(1,2,3))
    dice = (2 * inter + eps) / (union + eps)
    return dice.mean().item()


def few_shot_adapt_and_eval(model, task_root, n_shot=4, steps=10, lr_inner=1e-3, device=device):
    ds = SliceDataset(task_root, min_mask_sum=1.0, allow_empty=False)
    n = len(ds)
    if n <= n_shot:
        raise ValueError(f'Not enough slices with foreground in {task_root}')
    idxs = np.random.permutation(n)
    support_idx = idxs[:n_shot]
    query_idx   = idxs[n_shot:]
    xs, ys, xq_list, yq_list = [], [], [], []
    for i in support_idx:
        x, y = ds[i]; xs.append(x); ys.append(y)
    for i in query_idx:
        x, y = ds[i]; xq_list.append(x); yq_list.append(y)
    xs = torch.stack(xs).to(device)
    ys = torch.stack(ys).to(device)
    adapted = copy.deepcopy(model).to(device)
    opt = torch.optim.SGD(adapted.parameters(), lr=lr_inner)
    for _ in range(steps):
        opt.zero_grad()
        out_s = adapted(xs)
        if out_s.shape != ys.shape:
            ys = ys.view_as(out_s)
        loss = F.binary_cross_entropy(out_s, ys)
        loss.backward()
        opt.step()
    batch_size_q = 4
    all_preds, all_targets = [], []
    with torch.no_grad():
        for i in range(0, len(xq_list), batch_size_q):
            xb = torch.stack(xq_list[i:i+batch_size_q]).to(device)
            yb = torch.stack(yq_list[i:i+batch_size_q]).to(device)
            out_q = adapted(xb)
            if out_q.shape != yb.shape:
                yb = yb.view_as(out_q)
            all_preds.append(out_q.cpu())
            all_targets.append(yb.cpu())
            del xb, yb, out_q
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    preds = torch.cat(all_preds, dim=0)
    targets = torch.cat(all_targets, dim=0)
    d = dice_score(preds, targets)
    del adapted
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return d, preds, targets

# Run evaluation ONCE and store in-memory results list
results = []
for split_name, task_list in [('val', val_tasks), ('test', test_tasks)]:
    for task_root in task_list:
        task_id = os.path.basename(task_root)
        for shots in [1, 3, 5]:
            try:
                d, pq, yq = few_shot_adapt_and_eval(meta_model, task_root, n_shot=shots, steps=10, lr_inner=1e-3, device=device)
                results.append({'split': split_name, 'task': task_id, 'shots': shots, 'dice': float(d)})
                print(f'[{split_name}] {task_id} | {shots}-shot Dice: {d:.4f}')
            except Exception as e:
                print(f'[{split_name}] {task_id} | {shots}-shot FAILED: {e}')

[val] task_4 | 1-shot Dice: 0.0000
[val] task_4 | 3-shot Dice: 0.0000
[val] task_4 | 5-shot Dice: 0.0000
[val] task_6 | 1-shot Dice: 0.0000
[val] task_6 | 3-shot Dice: 0.0000
[val] task_6 | 5-shot Dice: 0.0000
[test] task_1 | 1-shot Dice: 0.0000
[test] task_1 | 3-shot Dice: 0.0000
[test] task_1 | 5-shot Dice: 0.0000
[test] task_8 | 1-shot Dice: 0.0000
[test] task_8 | 3-shot Dice: 0.0000
[test] task_8 | 5-shot Dice: 0.0000


## 9. Export config and results (no recomputation)

In [ ]:
experiment_config = {
    'meta_iters': meta_config['meta_iters'],
    'tasks_per_meta_batch': meta_config['tasks_per_meta_batch'],
    'k_support': meta_config['k_support'],
    'k_query': meta_config['k_query'],
    'lr_inner': meta_config['lr_inner'],
    'lr_meta': meta_config['lr_meta'],
    'allow_empty_meta': meta_config['allow_empty_meta'],
    'min_mask_sum': meta_config['min_mask_sum'],
    'few_shot_steps': 10,
    'few_shot_shots_list': [1, 3, 5],
    'device': str(device),
}

print('Current stored results:')
for r in results:
    print(r)

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
base_name = f'ems741_reptile_results_{ts}'
csv_path = base_name + '.csv'
json_path = base_name + '.json'
fieldnames = ['split', 'task', 'shots', 'dice']
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for row in results:
        writer.writerow(row)
with open(json_path, 'w') as f:
    json.dump({'config': experiment_config, 'results': results}, f, indent=2)
print('Saved without recomputing:')
print('  CSV :', csv_path)
print('  JSON:', json_path)

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 19)